# Auction Card Detector - Colab GPU training (3-class: card/label/slab)

**Run all cells top to bottom in one sitting** (free Colab recycles the runtime if idle).
First: `Runtime -> Change runtime type -> T4 GPU`.

Starts fresh from `yolo11n.pt` (class count changed from 1 to 3). Saves to a NEW
Drive folder `output_card_v2`, so your 1-class `output_card_v1` stays intact.

In [ ]:
# 1. Confirm GPU (should show Tesla T4)
!nvidia-smi

In [ ]:
# 2. Mount Drive, install Ultralytics, set config
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics

DRIVE_DIR = '/content/drive/MyDrive/card_detector'
RUN_NAME  = 'card_v2'   # 3-class run (v1 was the 1-class model)
EPOCHS    = 60
IMGSZ     = 640
BATCH     = 16

In [ ]:
# 3. Unzip dataset, locate data.yaml, repoint its path to Colab
import glob, shutil
from pathlib import Path

shutil.rmtree('/content/ds', ignore_errors=True)
!unzip -q "{DRIVE_DIR}/dataset.zip" -d /content/ds

yamls = glob.glob('/content/ds/**/data.yaml', recursive=True)
assert yamls, 'data.yaml not found - did you zip the dataset/ folder?'
DATA_YAML = yamls[0]
ds_root = str(Path(DATA_YAML).parent)
lines = Path(DATA_YAML).read_text().splitlines()
lines = [f'path: {ds_root}' if l.startswith('path:') else l for l in lines]
Path(DATA_YAML).write_text('\n'.join(lines) + '\n')
print('DATA_YAML =', DATA_YAML); print(Path(DATA_YAML).read_text())
for sp in ('train','val','test'):
    print(sp, len(glob.glob(f'{ds_root}/images/{sp}/*')), 'images')

In [ ]:
# 4. Train + evaluate + save to Drive (one cell)
import os, shutil
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
            patience=20, device=0, name=RUN_NAME, seed=0, plots=True)

save_dir = str(model.trainer.save_dir)
best_pt  = f'{save_dir}/weights/best.pt'

best = YOLO(best_pt)
mt = best.val(data=DATA_YAML, split='test', device=0)   # per-class table prints in the log
print('\n=================== TEST (action05, held-out raw seller) ===================')
print(f'OVERALL  mAP50={mt.box.map50:.3f}  P={mt.box.mp:.3f}  R={mt.box.mr:.3f}  mAP50-95={mt.box.map:.3f}')
# per-class (card is the only class present in test; label/slab have 0 instances here)
for i, name in enumerate(['card','label','slab']):
    try:
        p,r,ap50,ap = mt.box.class_result(i)
        print(f'  {name:5}: mAP50={ap50:.3f}  P={p:.3f}  R={r:.3f}')
    except Exception:
        print(f'  {name:5}: (no instances in test)')

out = f'{DRIVE_DIR}/output_{RUN_NAME}'
os.makedirs(out, exist_ok=True)
shutil.copy(best_pt, f'{out}/best.pt')
shutil.copytree(save_dir, f'{out}/run', dirs_exist_ok=True)
print('\nSaved best.pt + run to', out)